Load Dataset

In [37]:
import csv

def load_dataset(csv_file_path):
    entries = []
    with open(csv_file_path, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            entry = {
                "domain": row['Domain'],
                "model_prompt": row['Model prompt is generated from'],
                "prompt_without_context": row['Prompt without irrelevant context'],
                "prompt_with_context": row['Prompt with irrelevant context'],
                "small_llm_used": row['Small LLM used'],
                "small_llm_parameters": row['Small LLM parameters'],
                "response_without_context": row['Response from small LLM without irrelevant context'],
                "response_with_context": row['Response from small LLM with irrelevant context'],
                "correct_answer": row['Correct answer'],
                "llm_correct_without_context": row['LLM answer correctly without irrelevant context?'],
                "llm_correct_with_context": row['LLM answer correctly with irrelevant context?'],
                "context_effect": row['How did the irrelevant context affect the response?'],
                "use_in_final_dataset": row['Use example in final dataset? (yes or no)']
            }
            entries.append(entry)
    return entries


csv_file_path = r"custom_dataset.csv"
dataset = load_dataset(csv_file_path)
print(len(dataset))


28


Set up models

In [42]:
# 900mil Parameter models
model_900mil = [
    "gpt2-large",       # 762M parameters
]

# 1bil parameter models
model_1bil = [
    "gpt2-xl",          # 1.5B parameters
    # "funnel-transformer/large",
]

# 7bil parameter models
model_7bil = [
    # "EleutherAI/gpt-neox-7B",
    # "bigscience/bloomz-7b1",
]

all_models = model_900mil + model_1bil + model_7bil

Process each data entry

In [43]:
def process_entry(entry):
    # only needs prompt with and prompt without context and correct answer
    relevant_prompt = entry["prompt_without_context"]
    irrelevant_prompt = entry["prompt_with_context"]
    correct_answer = entry["correct_answer"]

    return {
        "relevant_prompt": relevant_prompt,
        "irrelevant_prompt": irrelevant_prompt,
        "correct_answer": correct_answer
    }

prompts_and_answers = []
for entry in dataset:
    prompts_and_answers.append(process_entry(entry))

In [44]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import torch
from sentence_transformers import SentenceTransformer, util

# Force CPU usage - for now
device = torch.device("cpu")

def chat(prompt, tokenizer, deivice, model):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=50, temperature=0.7, top_p=0.9, past_key_values=None)
    # Decode the output, skipping the first tokens (the prompt)
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # strip the prompt from the generated text by removing the prompt length from the output
    return generated_text[len(prompt):].strip()

similarity_model = SentenceTransformer('all-MiniLM-L6-v2')
def is_semantically_correct(response, correct_answer, threshold=0.8):
    # Compute embeddings
    response_embedding = similarity_model.encode(response, convert_to_tensor=True)
    correct_answer_embedding = similarity_model.encode(correct_answer, convert_to_tensor=True)

    # Compute cosine similarity
    similarity = util.cos_sim(response_embedding, correct_answer_embedding).item()

    # Check if similarity exceeds the threshold
    return similarity >= threshold, similarity # below 0.8 is not similar, above 0.8 is similar

# test multiple models
all_model_responses = []
# all_models = all_models[:1] # UNCOMMENT THIS LINE TO TEST ONLY THE FIRST MODEL
for model_name in all_models:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)

    model_responses = {
        "model_name": model_name,
        "responses": []
    }

    # prompts_and_answers = prompts_and_answers[:1] # UNCOMMENT THIS LINE TO TEST ONLY THE FIRST PROMPT
    for entry in prompts_and_answers:  # Loop over all prompts
        relevant_response = chat(entry["relevant_prompt"], tokenizer, device, model)
        irrelevant_response = chat(entry["irrelevant_prompt"], tokenizer, device, model)

        relevant_similarity, relevant_correct = is_semantically_correct(relevant_response, entry["correct_answer"])
        irrelevant_similarity, irrelevant_correct = is_semantically_correct(irrelevant_response, entry["correct_answer"])

        model_responses["responses"].append({
            "relevant_prompt": entry["relevant_prompt"],
            "irrelevant_prompt": entry["irrelevant_prompt"],
            "correct_answer": entry["correct_answer"],
            "relevant_response": relevant_response,
            "relevant_similarity": relevant_similarity,
            "relevant_correct": relevant_correct,
            "irrelevant_response": irrelevant_response,
            "irrelevant_similarity": irrelevant_similarity,
            "irrelevant_correct": irrelevant_correct
        })


    all_model_responses.append(model_responses)

for model_data in all_model_responses:
    print(model_data)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

{'model_name': 'gpt2-large', 'responses': [{'relevant_prompt': 'In a deck of 52 playing cards, you draw 2 cards randomly. The deck has 4 suits, and each suit has 13 cards. You are given the following information:\n\nThe first card you draw is not a spade.\nWhat is the probability that the second card drawn is a spade?', 'irrelevant_prompt': 'In a deck of 52 playing cards, you draw 2 cards randomly. The deck has 4 suits, and each suit has 13 cards. You are given the following information:\n\nThe first card you draw is not a spade.\nThe first card you draw is also not a heart.\nThe deck was shuffled using a Fisher-Yates algorithm, ensuring that each card has an equal chance of being drawn.\nYou notice that the deck has no jokers.\nWhat is the probability that the second card drawn is a spade?', 'correct_answer': 'Final answer: 13/51 or approximately 25.49%', 'relevant_response': 'What is the probability that the third card drawn is a spade?\n\nWhat is the probability that the fourth card

In [46]:
import csv

csv_data = []

for model_responses in all_model_responses:
    model_name = model_responses["model_name"]
    
    # Loop through each response entry
    for entry in model_responses["responses"]:
        relevant_response = entry["relevant_response"]
        irrelevant_response = entry["irrelevant_response"]
        relevant_similarity = entry["relevant_similarity"]
        irrelevant_similarity = entry["irrelevant_similarity"]
        relevant_correct = entry["relevant_correct"]
        irrelevant_correct = entry["irrelevant_correct"]

        # Store data for CSV if the conditions are met
        # if relevant_similarity and not irrelevant_similarity:
        csv_data.append({
            "model_name": model_name,
            "correct_answer": entry["correct_answer"],
            "relevant_prompt": entry["relevant_prompt"],
            "relevant_response": relevant_response,
            "relevant_correct": relevant_similarity,
            "relevant_similarity_score": relevant_correct,
            "irrelevant_prompt": entry["irrelevant_prompt"],
            "irrelevant_response": irrelevant_response,
            "irrelevant_correct": irrelevant_similarity,
            "irrelevant_similarity_score": irrelevant_correct,
        })

# Check if csv_data has any content before writing
if csv_data:
    csv_filename = "not_filtered_model_responses.csv"
    with open(csv_filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=csv_data[0].keys())
        writer.writeheader()
        writer.writerows(csv_data)

    print(f"Filtered responses have been written to {csv_filename}")
else:
    print("No data met the condition for saving to CSV.")
    
# Output all model responses
for model_data in all_model_responses:
    print(model_data)


Filtered responses have been written to not_filtered_model_responses.csv
{'model_name': 'gpt2-large', 'responses': [{'relevant_prompt': 'In a deck of 52 playing cards, you draw 2 cards randomly. The deck has 4 suits, and each suit has 13 cards. You are given the following information:\n\nThe first card you draw is not a spade.\nWhat is the probability that the second card drawn is a spade?', 'irrelevant_prompt': 'In a deck of 52 playing cards, you draw 2 cards randomly. The deck has 4 suits, and each suit has 13 cards. You are given the following information:\n\nThe first card you draw is not a spade.\nThe first card you draw is also not a heart.\nThe deck was shuffled using a Fisher-Yates algorithm, ensuring that each card has an equal chance of being drawn.\nYou notice that the deck has no jokers.\nWhat is the probability that the second card drawn is a spade?', 'correct_answer': 'Final answer: 13/51 or approximately 25.49%', 'relevant_response': 'What is the probability that the thi

In [ ]:
# part 2 - just the one bloom 7b model
# 900mil Parameter models
model_900mil = [
    # "gpt2-large",       # 762M parameters
]

# 1bil parameter models
model_1bil = [
    # "gpt2-xl",          # 1.5B parameters
]

# 7bil parameter models
model_7bil = [
    "bigscience/bloomz-7b1",
]

all_models = model_900mil + model_1bil + model_7bil

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import torch
from sentence_transformers import SentenceTransformer, util

# Force CPU usage - for now
device = torch.device("cpu")

def chat(prompt, tokenizer, deivice, model):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=50, temperature=0.7, top_p=0.9, past_key_values=None)
    # Decode the output, skipping the first tokens (the prompt)
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # strip the prompt from the generated text by removing the prompt length from the output
    return generated_text[len(prompt):].strip()

similarity_model = SentenceTransformer('all-MiniLM-L6-v2')
def is_semantically_correct(response, correct_answer, threshold=0.8):
    # Compute embeddings
    response_embedding = similarity_model.encode(response, convert_to_tensor=True)
    correct_answer_embedding = similarity_model.encode(correct_answer, convert_to_tensor=True)

    # Compute cosine similarity
    similarity = util.cos_sim(response_embedding, correct_answer_embedding).item()

    # Check if similarity exceeds the threshold
    return similarity >= threshold, similarity # below 0.8 is not similar, above 0.8 is similar

# test multiple models
all_model_responses = []
# all_models = all_models[:1] # UNCOMMENT THIS LINE TO TEST ONLY THE FIRST MODEL
for model_name in all_models:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)

    model_responses = {
        "model_name": model_name,
        "responses": []
    }

    # prompts_and_answers = prompts_and_answers[:1] # UNCOMMENT THIS LINE TO TEST ONLY THE FIRST PROMPT
    for entry in prompts_and_answers:  # Loop over all prompts
        relevant_response = chat(entry["relevant_prompt"], tokenizer, device, model)
        irrelevant_response = chat(entry["irrelevant_prompt"], tokenizer, device, model)

        relevant_similarity, relevant_correct = is_semantically_correct(relevant_response, entry["correct_answer"])
        irrelevant_similarity, irrelevant_correct = is_semantically_correct(irrelevant_response, entry["correct_answer"])

        model_responses["responses"].append({
            "relevant_prompt": entry["relevant_prompt"],
            "irrelevant_prompt": entry["irrelevant_prompt"],
            "correct_answer": entry["correct_answer"],
            "relevant_response": relevant_response,
            "relevant_similarity": relevant_similarity,
            "relevant_correct": relevant_correct,
            "irrelevant_response": irrelevant_response,
            "irrelevant_similarity": irrelevant_similarity,
            "irrelevant_correct": irrelevant_correct
        })


    all_model_responses.append(model_responses)

for model_data in all_model_responses:
    print(model_data)


import csv

csv_data = []

for model_responses in all_model_responses:
    model_name = model_responses["model_name"]
    
    # Loop through each response entry
    for entry in model_responses["responses"]:
        relevant_response = entry["relevant_response"]
        irrelevant_response = entry["irrelevant_response"]
        relevant_similarity = entry["relevant_similarity"]
        irrelevant_similarity = entry["irrelevant_similarity"]
        relevant_correct = entry["relevant_correct"]
        irrelevant_correct = entry["irrelevant_correct"]

        # Store data for CSV if the conditions are met
        # if relevant_similarity and not irrelevant_similarity:
        csv_data.append({
            "model_name": model_name,
            "correct_answer": entry["correct_answer"],
            "relevant_prompt": entry["relevant_prompt"],
            "relevant_response": relevant_response,
            "relevant_correct": relevant_similarity,
            "relevant_similarity_score": relevant_correct,
            "irrelevant_prompt": entry["irrelevant_prompt"],
            "irrelevant_response": irrelevant_response,
            "irrelevant_correct": irrelevant_similarity,
            "irrelevant_similarity_score": irrelevant_correct,
        })

# Check if csv_data has any content before writing
if csv_data:
    csv_filename = "not_filtered_model_responses2.csv"
    with open(csv_filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=csv_data[0].keys())
        writer.writeheader()
        writer.writerows(csv_data)

    print(f"Filtered responses have been written to {csv_filename}")
else:
    print("No data met the condition for saving to CSV.")
    
# Output all model responses
for model_data in all_model_responses:
    print(model_data)
